# RAG (Web Search) — `career_position` Annotation — Fine-grained

For each career entry, performs two Google searches (workplace + job title),
synthesises the top-3 snippets into a short context description, then feeds
that context into the classification prompt.

**Model:** TBD — set `MODEL_ID` below  
**Requires:** `OPENAI_API_KEY` (or LM Studio), `GOOGLE_API_KEY`, `GOOGLE_CSE_ID` in `.env`

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import json
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from openai import OpenAI
from tqdm import tqdm

from corex_eval import evaluate, load_inputs, load_training_data

# ── Model (TBD: set once benchmark results are in) ──────────────────────────
MODEL_ID       = "TBD"          # e.g. "gpt-5.3-chat-latest" or LM Studio id
LMSTUDIO_URL   = None           # set to "http://localhost:1234/v1" for local
TEMPERATURE    = 0
MAX_TOKENS     = 64
SEED           = 42

# ── Google Custom Search ────────────────────────────────────────────────────
GOOGLE_API_KEY = os.environ["GOOGLE_API_KEY"]
GOOGLE_CSE_ID  = os.environ["GOOGLE_CSE_ID"]
N_SEARCH_RESULTS = 3

# ── Cache ───────────────────────────────────────────────────────────────────
CACHE_DIR      = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
SEARCH_CACHE_FILE   = CACHE_DIR / "search_cache.json"
SYNTHESIS_CACHE_FILE = CACHE_DIR / "synthesis_cache.json"

client = OpenAI(
    base_url=LMSTUDIO_URL,
    api_key=os.environ.get("OPENAI_API_KEY", "lm-studio"),
)
print(f"Model: {MODEL_ID}")

## 2. Cache helpers

In [ ]:
def _load_cache(path: Path) -> dict:
    if path.exists():
        with open(path) as f:
            return json.load(f)
    return {}

def _save_cache(cache: dict, path: Path) -> None:
    with open(path, "w") as f:
        json.dump(cache, f)

search_cache    = _load_cache(SEARCH_CACHE_FILE)
synthesis_cache = _load_cache(SYNTHESIS_CACHE_FILE)
print(f"Loaded {len(search_cache)} cached searches, {len(synthesis_cache)} cached syntheses")

## 3. Load valid codes & system prompt

In [ ]:
train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["career_position"],
)
valid_codes = sorted(train_df["career_position"].dropna().unique().tolist())
codes_text  = "\n".join(f"  {c}" for c in valid_codes)

CLASSIFICATION_SYSTEM_PROMPT = f"""You are an expert political scientist coding political career histories using the CoREx codebook.

Task: given a career entry (job title, workplace, and optional context), assign the single most appropriate code.
Respond with ONLY the exact code label as listed below — nothing else.

Valid codes:
{codes_text}"""

SYNTHESIS_SYSTEM_PROMPT = """You are a concise research assistant. Based on the provided web search snippets,
write 1-2 sentences describing the subject. Focus on its sector, role, or function.
If the snippets are not relevant or insufficient, respond with exactly: NA"""

print(f"{len(valid_codes)} valid codes loaded")

## 4. Load test inputs

In [ ]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df[["case_id", "spell_index", "job_description_label", "workplace_label"]].head()

## 5. Web search & synthesis (cached)

In [ ]:
def google_search(query: str) -> list[str]:
    """Return top-N snippet strings from Google Custom Search. Cached."""
    if query in search_cache:
        return search_cache[query]
    try:
        resp = requests.get(
            "https://www.googleapis.com/customsearch/v1",
            params={"q": query, "key": GOOGLE_API_KEY, "cx": GOOGLE_CSE_ID, "num": N_SEARCH_RESULTS},
            timeout=10,
        )
        items = resp.json().get("items", [])
        snippets = [item.get("snippet", "") for item in items]
    except Exception as e:
        print(f"Search error for '{query}': {e}")
        snippets = []
    search_cache[query] = snippets
    _save_cache(search_cache, SEARCH_CACHE_FILE)
    time.sleep(0.1)   # gentle rate limiting
    return snippets


def synthesise(subject: str, snippets: list[str], context: str = "") -> str:
    """Ask the LLM to summarise search snippets into 1-2 sentences. Cached."""
    cache_key = f"{subject}||{context}"
    if cache_key in synthesis_cache:
        return synthesis_cache[cache_key]
    if not snippets:
        synthesis_cache[cache_key] = "NA"
        _save_cache(synthesis_cache, SYNTHESIS_CACHE_FILE)
        return "NA"
    snippets_text = "\n".join(f"[{i+1}] {s}" for i, s in enumerate(snippets))
    user_msg = f"Subject: {subject}\n"
    if context:
        user_msg += f"Context: {context}\n"
    user_msg += f"\nSearch snippets:\n{snippets_text}"
    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": SYNTHESIS_SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=0,
            max_tokens=128,
        )
        result = resp.choices[0].message.content.strip()
    except Exception as e:
        print(f"Synthesis error for '{subject}': {e}")
        result = "NA"
    synthesis_cache[cache_key] = result
    _save_cache(synthesis_cache, SYNTHESIS_CACHE_FILE)
    return result


def get_context(job_title: str, workplace: str) -> tuple[str, str]:
    """Return (workplace_description, role_description) for a career entry."""
    wp_snippets   = google_search(f"{workplace} organization political")
    job_snippets  = google_search(f"{job_title} {workplace} role responsibilities")
    wp_desc       = synthesise(workplace, wp_snippets)
    role_desc     = synthesise(f"{job_title} at {workplace}", job_snippets, context=workplace)
    return wp_desc, role_desc

## 6. Inference

In [ ]:
def parse_prediction(text: str, valid_codes: list[str]) -> str:
    text = text.strip()
    if text in valid_codes:
        return text
    for code in valid_codes:
        if text.startswith(code.split(" = ")[0].strip()):
            return code
    text_lower = text.lower()
    for code in valid_codes:
        if code.lower() in text_lower:
            return code
    return text


predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    job_title = row["job_description_label"]
    workplace = row.get("workplace_label", "") or ""

    wp_desc, role_desc = get_context(job_title, workplace)

    context_block = ""
    if wp_desc and wp_desc != "NA":
        context_block += f"\nAbout the workplace: {wp_desc}"
    if role_desc and role_desc != "NA":
        context_block += f"\nAbout the role: {role_desc}"

    user_msg = f"Job title: {job_title}\nWorkplace: {workplace or 'unknown'}{context_block}"

    try:
        resp = client.chat.completions.create(
            model=MODEL_ID,
            messages=[
                {"role": "system", "content": CLASSIFICATION_SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
            seed=SEED,
        )
        raw = resp.choices[0].message.content or ""
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""

    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

## 7. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})